In [ ]:
!pip install torch transformers accelerate sentencepiece pandas tqdm

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    pipeline
)


In [ ]:
############################################
# CONFIG
############################################
INPUT_CSV = "aws_review_data/test.csv"
OUTPUT_CSV = "aws_review_data/test_translated_scored.csv"
BATCH_SIZE = 16
MAX_INPUT_LEN = 512
MAX_NEW_TOKENS = 256

TRANSLATION_MODEL = "tencent/Hunyuan-MT-Chimera-7B"
SENTIMENT_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(DEVICE)

In [ ]:
############################################
# LOAD TRANSLATION MODEL
############################################
print("Loading Hunyuan-MT-Chimera-7B...")

mt_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL,
    trust_remote_code=True
)

mt_model = AutoModelForCausalLM.from_pretrained(
    TRANSLATION_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
############################################
# LOAD SENTIMENT MODEL
############################################
print("Loading sentiment model...")

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    device=0 if DEVICE == "cuda" else -1
)

In [ ]:
############################################
# TRANSLATION FUNCTION
############################################
def translate_to_english(texts, src_lang):
    """
    Translate a batch of texts from src_lang to English using
    Hunyuan-MT-Chimera-7B (decoder-only).

    texts: List[str]
    src_lang: str (de, fr, es, ja, zh, en)
    """

    prompts = [
        f"<|{src_lang}|> <|en|> {text}" for text in texts
    ]

    inputs = mt_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    # 🔧 IMPORTANT FIX:
    # Hunyuan does NOT use token_type_ids
    inputs.pop("token_type_ids", None)

    inputs = inputs.to(mt_model.device)

    with torch.no_grad():
        outputs = mt_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256,
            num_beams=4,
            do_sample=False
        )

    decoded = mt_tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    # Remove the prompt prefix from generated text
    translations = [
        out[len(prompt):].strip()
        if out.startswith(prompt) else out.strip()
        for out, prompt in zip(decoded, prompts)
    ]

    return translations

In [ ]:
############################################
# SENTIMENT SCORING FUNCTION
############################################
def get_sentiment_scores(texts):
    """
    Returns integer sentiment scores from 1 to 5
    Safely handles long texts by truncating to BERT max length (512)
    """

    results = sentiment_pipeline(
        texts,
        batch_size=32,
        truncation=True,     
        max_length=512       
    )

    scores = []
    for r in results:
        # label format: "1 star", "2 stars", ...
        score = int(r["label"].split()[0])
        scores.append(score)
    
    return scores

In [ ]:
############################################
# MAIN PIPELINE
############################################

def process_reviews(df):
    translated_reviews = []
    sentiment_scores = []

    total_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Starting processing: {len(df)} reviews, {total_batches} batches")

    for batch_idx in tqdm(range(0, len(df), BATCH_SIZE), desc="Processing batches"):
        batch = df.iloc[batch_idx:batch_idx + BATCH_SIZE]

        texts = batch["review_body"].fillna("").tolist()
        langs = batch["language"].tolist()

        translations_map = {}

        # -------------------------
        # Translation (per language)
        # -------------------------
        for lang in sorted(set(langs)):
            indices = [i for i, l in enumerate(langs) if l == lang]
            lang_texts = [texts[i] for i in indices]

            print(f"[Batch {batch_idx}] Translating {len(lang_texts)} reviews (lang={lang})")

            if lang == "en":
                translated = lang_texts
            else:
                translated = translate_to_english(lang_texts, lang)

            for i, t in zip(indices, translated):
                translations_map[i] = t

            print(f"[Batch {batch_idx}] Translation done (lang={lang})")

        ordered_translations = [translations_map[i] for i in range(len(batch))]

        # -------------------------
        # Sentiment scoring
        # -------------------------
        print(f"[Batch {batch_idx}] Scoring sentiment for {len(ordered_translations)} reviews")

        scores = []
        for j in range(0, len(ordered_translations), 32):
            sub_batch = ordered_translations[j:j + 32]
            sub_scores = get_sentiment_scores(sub_batch)
            scores.extend(sub_scores)

        print(f"[Batch {batch_idx}] Sentiment scoring completed")

        translated_reviews.extend(ordered_translations)
        sentiment_scores.extend(scores)

    print("Processing completed successfully ✅")

    return translated_reviews, sentiment_scores

In [ ]:
############################################
# RUN
############################################
if __name__ == "__main__":
    print("Reading CSV...")
    df = pd.read_csv(INPUT_CSV)

    print("Translating and scoring reviews...")
    translated_reviews, sentiment_scores = process_reviews(df)

    df["review_body_en"] = translated_reviews
    df["sentiment_score_1_to_5"] = sentiment_scores

    print(f"Saving output to {OUTPUT_CSV}")
    df.to_csv(OUTPUT_CSV, index=False)

    print("Done")